# Notebook 03 — Integrals and Area Under a Curve

**Module:** 02 — Mathematics for Biology  
**Notebook:** 03 of 12  
**Prerequisites:** NB02  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

Integrals quantify *accumulated change*. In pharmacokinetics, the area under the
drug concentration curve (AUC) determines total drug exposure — the actual therapeutic
dose the body receives. In genomics, the integral of a signal across a genomic window
defines peak intensity in ChIP-seq. The derivative tells you the rate; the integral
tells you the total.

---
## Step 2 — Intuition

An integral is the limit of a sum: divide the x-axis into thin strips, compute the
area of each (height × width), and add them all up. As the strips get thinner, the
sum converges to the exact area under the curve.

Derivatives and integrals are inverses: if $F'(x) = f(x)$, then
$\int_a^b f(x)\,dx = F(b) - F(a)$. This is the Fundamental Theorem of Calculus.

---
## Step 3 — Biological Background

**Pharmacokinetics — drug concentration over time:**

After an intravenous dose $D$, drug concentration decays exponentially:
$$C(t) = C_0 e^{-k_e t}$$

where $k_e$ is the elimination rate constant and $C_0 = D/V_d$ (volume of distribution).

**AUC (area under the concentration-time curve):**
$$\text{AUC}_{0 \to \infty} = \int_0^\infty C(t)\,dt = \frac{C_0}{k_e}$$

AUC is the key pharmacokinetic parameter: a drug must exceed a therapeutic threshold
AUC to be effective and stay below a toxicity threshold AUC to be safe.

**Half-life:** $t_{1/2} = \ln(2)/k_e$ — the time for concentration to halve.

---
## Step 4 — Mathematical Explanation

**Riemann sum (left endpoint):**
$$\int_a^b f(x)\,dx \approx \sum_{i=0}^{n-1} f(x_i) \cdot \Delta x$$

where $\Delta x = (b-a)/n$ and $x_i = a + i \cdot \Delta x$.

**Trapezoid rule** (better accuracy):
$$\int_a^b f(x)\,dx \approx \sum_{i=0}^{n-1} \frac{f(x_i) + f(x_{i+1})}{2} \cdot \Delta x$$

**Analytical result for exponential decay:**
$$\int_0^\infty C_0 e^{-k_e t}\,dt = \left[-\frac{C_0}{k_e} e^{-k_e t}\right]_0^\infty = \frac{C_0}{k_e}$$

---
## Step 5 — Computational Explanation

| Method | NumPy/SciPy API | Error order | Notes |
|--------|----------------|-------------|-------|
| Riemann sum (left) | manual loop or `np.sum` | O(h) | Simple to implement; least accurate |
| Trapezoid rule | `np.trapz` | O(h²) | Good default for tabular data |
| Simpson's rule | `scipy.integrate.simpson` | O(h⁴) | Better for smooth functions |
| Gaussian quadrature | `scipy.integrate.quad` | exponential | Use for analytical functions; most accurate |

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
from scipy.integrate import quad, simpson
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Riemann sums from scratch
def riemann_sum(f, a: float, b: float, n: int, method: str = "midpoint") -> float:
    """
    Compute a Riemann sum approximation of ∫_a^b f(x) dx.

    Parameters
    ----------
    f : callable
    a, b : float — integration limits
    n : int — number of subintervals
    method : 'left', 'right', or 'midpoint'

    Returns
    -------
    float — approximate integral
    """
    dx = (b - a) / n
    offsets = {"left": 0.0, "right": 1.0, "midpoint": 0.5}
    if method not in offsets:
        raise ValueError(f"method must be one of {list(offsets)}")
    x = a + (np.arange(n) + offsets[method]) * dx
    return np.sum(f(x)) * dx

# Test: ∫_0^1 x^2 dx = 1/3
f_sq = lambda x: x**2
true_val = 1/3
for method in ["left", "right", "midpoint"]:
    approx = riemann_sum(f_sq, 0, 1, n=1000, method=method)
    print(f"  {method:<10}  n=1000  result={approx:.6f}  error={abs(approx - true_val):.2e}")

In [ ]:
# Cell 6.2 — Drug AUC: exponential decay
C0 = 100.0   # mg/L — initial concentration
ke = 0.3     # per hour — elimination rate

def concentration(t: np.ndarray) -> np.ndarray:
    return C0 * np.exp(-ke * t)

# Analytical AUC from 0 to infinity
AUC_analytical = C0 / ke
print(f"Analytical AUC (0→∞): {AUC_analytical:.4f} mg·h/L")
print(f"Half-life: {np.log(2)/ke:.2f} hours")

# Numerical AUC from 0 to 24 hours (practical limit)
t_obs = np.linspace(0, 24, 1000)
AUC_trap   = np.trapz(concentration(t_obs), t_obs)
AUC_simp   = simpson(concentration(t_obs), x=t_obs)
AUC_quad, _ = quad(concentration, 0, np.inf)  # exact

print(f"\nNumerical AUC (0→24h, n=1000):")
print(f"  Trapezoid:      {AUC_trap:.4f}")
print(f"  Simpson:        {AUC_simp:.4f}")
print(f"  scipy.quad(0→∞):{AUC_quad:.4f}  (exact)")

In [ ]:
# Cell 6.3 — Convergence: error vs. number of intervals
n_vals = [10, 50, 100, 500, 1000, 5000]
errors_left, errors_trap = [], []

for n in n_vals:
    t_n = np.linspace(0, 24, n+1)
    c_n = concentration(t_n)
    # Left Riemann
    dx = 24/n
    left = np.sum(c_n[:-1]) * dx
    errors_left.append(abs(left - AUC_quad))
    # Trapezoid
    trap = np.trapz(c_n, t_n)
    errors_trap.append(abs(trap - AUC_quad))

print(f"{'n':>8} {'Left error':>15} {'Trapezoid error':>18}")
for n, el, et in zip(n_vals, errors_left, errors_trap):
    print(f"  {n:>6}   {el:>12.2e}     {et:>12.2e}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Drug concentration curve with shaded AUC
t_plot = np.linspace(0, 24, 500)
C_plot = concentration(t_plot)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: concentration curve with AUC shaded
ax = axes[0]
ax.plot(t_plot, C_plot, color="steelblue", lw=2)
ax.fill_between(t_plot, C_plot, alpha=0.25, color="steelblue", label=f"AUC₀₋₂₄ = {AUC_trap:.1f} mg·h/L")
ax.axhline(C0/2, color="gray", ls="--", alpha=0.7, label=f"t½ = {np.log(2)/ke:.1f} h")
ax.axvline(np.log(2)/ke, color="gray", ls="--", alpha=0.7)
ax.set_xlabel("Time (hours)"); ax.set_ylabel("Concentration (mg/L)")
ax.set_title("Drug concentration — AUC shaded")
ax.legend()

# Right: convergence
ax = axes[1]
ax.loglog(n_vals, errors_left, "o-", color="tomato", label="Left Riemann")
ax.loglog(n_vals, errors_trap, "s-", color="steelblue", label="Trapezoid")
ax.set_xlabel("Number of intervals (n)")
ax.set_ylabel("|Error|")
ax.set_title("Integration error vs. n")
ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `trapezoid_auc(t_obs, C_obs)` that takes observation times and concentrations
   as arrays and returns the AUC using the trapezoid rule. Add it to `compbio_utils/stats.py`.
2. Verify the Fundamental Theorem: compute `∫_0^x sin(t) dt` numerically for `x = π`
   using Riemann sums, and confirm it equals `1 - cos(π)` (the analytical result).
3. A drug has $C_0 = 50$ mg/L and $k_e = 0.1$/h. What is its AUC? Half-life?
   Compute both analytically and numerically. Report the relative error.
4. Looking at the convergence plot: what is the approximate slope of each method
   on the log-log scale? Does this match the expected O(h) and O(h²) rates?

---
## Quiz — Active Recall

1. What is AUC in pharmacokinetics? What does it measure?
2. What is the Fundamental Theorem of Calculus in one sentence?
3. Write the trapezoid rule formula for n intervals.
4. Why is `scipy.integrate.quad` more accurate than a Riemann sum for a smooth function?
5. If a drug's half-life is 4 hours, what is $k_e$? Derive it.

---
## Reflection

**Date completed:** ____________________

> *[Is trapezoid_auc in compbio_utils? Could you compute AUC from scratch on a whiteboard?]*

---
*Next: `04_exponential_logistic_growth.ipynb`*